In [1]:
#all necessary imports
import numpy as np
import pandas as pd
import polars as pl
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
import os

# TA indicators
import ta
from ta.trend import PSARIndicator
 
# Data Normalization and Scaling
from sklearn.preprocessing import MinMaxScaler
import numpy as np

# Scikit-learn utilities
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

# hyperparameter optimization
import optuna
import torch
import torch.nn as nn
import numpy as np
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import f1_score
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Training device:", device)

print("Torch version:", torch.__version__)
print("Polars version:", pl.__version__)
print("TQDM imported successfully!")
print("NumPy version:", np.__version__)
print("Pandas version:", pd.__version__)
print("TA library imported successfully!")
print("Optuna version:",optuna.__version__)

Training device: cuda
Torch version: 2.5.1+cu121
Polars version: 0.20.26
TQDM imported successfully!
NumPy version: 1.26.4
Pandas version: 2.2.2
TA library imported successfully!
Optuna version: 4.8.0


c:\Users\User\miniconda3\envs\quant_dl\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
#special case company
"""
"4348"
'2285', '4081', '2286', '4340', '4342', '2283', 
'2284', '1323', '4009', '4334', '4084', '4261', 
'4012', '1830', '4333', '1835', '4165', '4347', 
'2382', '2223', '4163', '4262', '4083', '4164', 
'1834', '1303', '2081', '4162', '7204', '7202', 
'2281', '1833', '4143', '4142', '4263', '1831', 
'4051', '1322', '1182', '6017', '4161', '4031', 
'4071', '4322', '4264', '4016', '4072', '4191', 
'4332', '2381', '6014', '4017', '4015', '4324', 
'6015', '4336', '4338', '4082', '4144', '4014', 
'1111', '4291', '4339', '4013', '2282', '4325', 
'1304', '7203', '4018', '2084', '7200', '2287', 
'4331', '8313', '4193', '1183', '4192', '4345', 
'4344', '6016', '2082', '2083', '6013', '3007',

# failed training
"4003","1010", "3030","8200","1150", "2020", "1140","7020",
    "8100", "2060", "1180", "3080", 
    "2150", "4260",
    "3020", "1320", "4007", "3040",
    "8280", "6070",  "2270", "3003",  

"""


"""

# done training (4 Output already)
"3090","8300","1050"
"2050", "4001",

"1030","8170","3030","1020","3010","2060", "1180"

"4250","2280","4020","3080"

"8160"

# on hold 
"1214", "8010", "3002", "3050", "8030",  "3005",
    "2230", "4280",  "4004", "8210",  "1212",
        "1010", "8100", "1120","4002", "8012",
        "4150", "8070","3004", "2080",
    "3060", "1302", "4300", "4090", "6001", "6004",
    "7040", "2310", "2300", "8230", "4200", "4100",
"2190", "2250", "1211", "8180", "2290", "2240", "4180",
"8240", "4210", "8040", "8310", "4050", "4040", "1210", 
"""



symbols = [
    
   
    "4080", "2010",
    "4170", "4240"
    ]



dfs = {}

for s in symbols:
    filepath = f"{s}_daily.csv"
    
    if not os.path.exists(filepath):
        print(f"  WARNING — {filepath} not found, skipping.")
        continue
    
    df = pd.read_csv(filepath)
    dfs[s] = df
    print(f"\n── {s} ──────────────────────────")
    print(f"  Shape : {df.shape}")
    print(f"  Head  :\n{df.head()}")
    print(f"  Dtypes:\n{df.dtypes}")
    print(f"  Nulls :\n{df.isnull().sum()}")

print(f"\n── Loaded {len(dfs)}/{len(symbols)} tickers ──")


── 4080 ──────────────────────────
  Shape : (5544, 46)
  Head  :
              datetime  ticker  open_price  high_price  low_price  \
0  2004-02-24 16:00:00    4080    26.34808    28.93286   26.01456   
1  2004-02-25 16:00:00    4080    29.05793    29.05793   27.51540   
2  2004-02-28 16:00:00    4080    29.14131    32.01792   28.76610   
3  2004-02-29 16:00:00    4080    31.60102    33.68552   31.05905   
4  2004-03-01 16:00:00    4080    32.01792    32.85172   30.68384   

   close_price        volume     EMA_10     EMA_20  EMA_ratio  ...  \
0     28.93286  7.907250e+06  26.054724  24.713854   1.170714  ...   
1     28.09906  2.812827e+06  26.426421  25.036255   1.122335  ...   
2     31.55933  6.901037e+06  27.359677  25.657500   1.230024  ...   
3     32.05961  5.093176e+06  28.214211  26.267225   1.220518  ...   
4     31.39257  2.505643e+06  28.792094  26.755353   1.173319  ...   

   volume_lag_3  volume_lag_4  volume_lag_5  above_ema10  above_ema20  \
0  1.471486e+06  7.29743

In [3]:
for symbol in symbols:
    df = pd.read_csv(f'{symbol}_daily.csv', parse_dates=["datetime"])
    
    # add your lag features + volume_pct_change_log here same as your pipeline
    for lag in range(1, 6):
        df[f"close_lag_{lag}"]   = df["close_price"].shift(lag)
        df[f"returns_lag_{lag}"] = df["close_price"].pct_change(1).shift(lag) * 100 
        df[f"volume_lag_{lag}"]  = df["volume"].shift(lag)

    # Regime and momentum persistence features
    df["above_ema10"] = (df["close_price"] > df["EMA_10"]).astype(float)
    df["above_ema20"] = (df["close_price"] > df["EMA_20"]).astype(float)
    df["roc5_pos"]    = (df["ROC_5"] > 0).astype(float)
    df["roc20_pos"]   = (df["ROC_20"] > 0).astype(float)
    daily_up = df["close_price"].diff(1).gt(0).astype(int)
    streak_id = (daily_up != daily_up.shift()).cumsum()
    df["up_streak"] = (daily_up.groupby(streak_id).cumcount() + 1) * daily_up
    df["volume_log"]            = np.log1p(df["volume"])
    df["volume_pct_change_log"] = df["volume_log"].diff(1).shift(-1)
    df = df.dropna().reset_index(drop=True)

    # outlier removal
    for col, sigma in {'price_pct_change': 3, 'volume_pct_change_log': 2}.items():
        mean, std = df[col].mean(), df[col].std()
        df = df[df[col].between(mean - sigma * std, mean + sigma * std)]
    df = df.reset_index(drop=True)

    # split
    n = len(df)
    train_df = df.iloc[:int(n * 0.75)]
    val_df   = df.iloc[int(n * 0.75):int(n * 0.875)]
    test_df  = df.iloc[int(n * 0.875):]

    # scale
    target_scaler = StandardScaler()
    train_y = target_scaler.fit_transform(train_df[['price_pct_change', 'volume_pct_change_log']])
    val_y   = target_scaler.transform(val_df[['price_pct_change', 'volume_pct_change_log']])

    print(f"\n{symbol}")
    print(f"  Raw train UP%      : {(train_df['price_pct_change'] > 0).mean():.2%}")
    print(f"  Raw val UP%        : {(val_df['price_pct_change'] > 0).mean():.2%}")
    print(f"  Scaled train mean  : {train_y[:, 0].mean():.4f}")  # should be ~0
    print(f"  Scaled train std   : {train_y[:, 0].std():.4f}")   # should be ~1
    print(f"  Scaled val mean    : {val_y[:, 0].mean():.4f}")    # if far from 0, val is different regime
    print(f"  Scaled val std     : {val_y[:, 0].std():.4f}")
    print(f"  Train date range   : {train_df['datetime'].min().date()} → {train_df['datetime'].max().date()}")
    print(f"  Val date range     : {val_df['datetime'].min().date()} → {val_df['datetime'].max().date()}")


4080
  Raw train UP%      : 46.64%
  Raw val UP%        : 41.34%
  Scaled train mean  : -0.0000
  Scaled train std   : 1.0000
  Scaled val mean    : -0.1077
  Scaled val std     : 0.8285
  Train date range   : 2004-03-03 → 2020-12-14
  Val date range     : 2020-12-15 → 2023-09-06

2010
  Raw train UP%      : 44.32%
  Raw val UP%        : 49.29%
  Scaled train mean  : -0.0000
  Scaled train std   : 1.0000
  Scaled val mean    : -0.0199
  Scaled val std     : 0.7601
  Train date range   : 2002-02-06 → 2020-04-09
  Val date range     : 2020-04-12 → 2023-04-26

4170
  Raw train UP%      : 47.14%
  Raw val UP%        : 43.92%
  Scaled train mean  : 0.0000
  Scaled train std   : 1.0000
  Scaled val mean    : -0.0459
  Scaled val std     : 0.7424
  Train date range   : 2004-03-03 → 2020-11-16
  Val date range     : 2020-11-17 → 2023-08-23

4240
  Raw train UP%      : 46.88%
  Raw val UP%        : 41.01%
  Scaled train mean  : -0.0000
  Scaled train std   : 1.0000
  Scaled val mean    : -0.13

In [4]:
import pickle
import os

SEQ_LEN_DEFAULT = 30
prepared = {}
scalers  = {}

FEATURE_COLS = [
    "open_price", "high_price", "low_price",
    "close_price", "volume",
    "EMA_10", "EMA_20", "EMA_ratio", "MACD_hist",
    "RSI_14", "ROC_5", "ROC_10", "ROC_20",
    "ATR_14", "ATR_ratio", "BB_pct", "realized_vol",
    "OBV", "OBV_momentum", "volume_ratio",
    "volume_surge", "MFI_14",
    "close_lag_1", "close_lag_2", "close_lag_3", "close_lag_4", "close_lag_5",
    "returns_lag_1", "returns_lag_2", "returns_lag_3", "returns_lag_4", "returns_lag_5",
    "volume_lag_1", "volume_lag_2", "volume_lag_3", "volume_lag_4", "volume_lag_5",
    "above_ema10", "above_ema20", "roc5_pos", "roc20_pos", "up_streak",
]

TARGET_COLS = ["price_pct_change", "volume_pct_change_log", "high_pct_change", "low_pct_change"]


def _make_seqs(X, y, seq_len):
    """Simple sliding-window builder."""
    Xs, ys, ws = [], [], []
    for i in range(len(X) - seq_len):
        Xs.append(X[i : i + seq_len])
        ys.append(y[i + seq_len])
        ws.append(1.0)
    return np.array(Xs), np.array(ys), np.array(ws, dtype=np.float32)


for symbol in symbols:
    filepath = f"{symbol}_daily.csv"
    if not os.path.exists(filepath):
        print(f"  WARNING — {filepath} not found, skipping.")
        continue

    df = pd.read_csv(filepath, parse_dates=["datetime"])

    # ── Lag features ──────────────────────────────────────────────────────────
    for lag in range(1, 6):
        df[f"close_lag_{lag}"]   = df["close_price"].shift(lag)
        df[f"returns_lag_{lag}"] = df["close_price"].pct_change(1).shift(lag) 
        df[f"volume_lag_{lag}"]  = df["volume"].shift(lag)

    # ── Regime / persistence features ─────────────────────────────────────────
    df["above_ema10"] = (df["close_price"] > df["EMA_10"]).astype(float)
    df["above_ema20"] = (df["close_price"] > df["EMA_20"]).astype(float)
    df["roc5_pos"]    = (df["ROC_5"] > 0).astype(float)
    df["roc20_pos"]   = (df["ROC_20"] > 0).astype(float)

    daily_up  = df["close_price"].diff(1).gt(0).astype(int)
    streak_id = (daily_up != daily_up.shift()).cumsum()
    df["up_streak"] = (daily_up.groupby(streak_id).cumcount() + 1) * daily_up

    # ── Volume target (log-diff shifted -1) ───────────────────────────────────
    df["volume_log"]            = np.log1p(df["volume"])
    df["volume_pct_change_log"] = df["volume_log"].diff(1).shift(-1)

    # ── High / Low % change targets (shifted -1) ──────────────────────────────
    df["high_pct_change"] = df["high_price"].pct_change(1).shift(-1) 
    df["low_pct_change"]  = df["low_price"].pct_change(1).shift(-1) 

    df = df.dropna().reset_index(drop=True)

    # ── Clip outliers ─────────────────────────────────────────────────────────
    for col, sigma in {
        "price_pct_change": 3,
        "volume_pct_change_log": 2, 
        "high_pct_change":3,
        "low_pct_change":3
        }.items():
        mean, std = df[col].mean(), df[col].std()
        df = df[df[col].between(mean - sigma * std, mean + sigma * std)]
    df = df.reset_index(drop=True)

    # ── Chronological 70 / 15 / 15 split ─────────────────────────────────────
    n        = len(df)
    train_df = df.iloc[:int(n * 0.75)]
    val_df   = df.iloc[int(n * 0.75):int(n * 0.875)]
    test_df  = df.iloc[int(n * 0.875):]

    train_X_raw = train_df[FEATURE_COLS].values.astype(np.float32)
    val_X_raw   = val_df[FEATURE_COLS].values.astype(np.float32)
    test_X_raw  = test_df[FEATURE_COLS].values.astype(np.float32)

    train_y_raw = train_df[TARGET_COLS].values.astype(np.float32)
    val_y_raw   = val_df[TARGET_COLS].values.astype(np.float32)
    test_y_raw  = test_df[TARGET_COLS].values.astype(np.float32)

    # ── Fit scalers on train only ─────────────────────────────────────────────
    feature_scaler = StandardScaler()
    target_scaler  = StandardScaler()

    train_X_scaled = feature_scaler.fit_transform(train_X_raw)
    val_X_scaled   = feature_scaler.transform(val_X_raw)
    test_X_scaled  = feature_scaler.transform(test_X_raw)

    train_y_scaled = target_scaler.fit_transform(train_y_raw)
    val_y_scaled   = target_scaler.transform(val_y_raw)
    test_y_scaled  = target_scaler.transform(test_y_raw)

    # ── Save scalers ──────────────────────────────────────────────────────────
    os.makedirs(f"models/{symbol}", exist_ok=True)
    with open(f"models/{symbol}/{symbol}_feature_scaler.pkl", "wb") as f:
        pickle.dump(feature_scaler, f)
    with open(f"models/{symbol}/{symbol}_target_scaler.pkl", "wb") as f:
        pickle.dump(target_scaler, f)

    # ── Pre-build sequences at default SEQ_LEN ────────────────────────────────
    X_train, y_train, _ = _make_seqs(train_X_scaled, train_y_scaled, SEQ_LEN_DEFAULT)
    X_val,   y_val,   _ = _make_seqs(val_X_scaled,   val_y_scaled,   SEQ_LEN_DEFAULT)
    X_test,  y_test,  _ = _make_seqs(test_X_scaled,  test_y_scaled,  SEQ_LEN_DEFAULT)

    prepared[symbol] = {
        # Scaled 2D arrays → used by Optuna cell + training loop (seq_len chosen per trial)
        "train_X_scaled": train_X_scaled,
        "train_y_scaled": train_y_scaled,
        "val_X_scaled":   val_X_scaled,
        "val_y_scaled":   val_y_scaled,
        "test_X_scaled":  test_X_scaled,
        "test_y_scaled":  test_y_scaled,
        # Pre-built 3D sequences at SEQ_LEN_DEFAULT → used by DataLoader cell
        "X_train": X_train,  "y_train": y_train,
        "X_val":   X_val,    "y_val":   y_val,
        "X_test":  X_test,   "y_test":  y_test,
    }
    scalers[symbol] = {
        "feature_scaler": feature_scaler,
        "target_scaler":  target_scaler,
    }

    print(f"\n── {symbol} {'─' * 60}")
    print(f"  Rows  → train:{len(train_df)}  val:{len(val_df)}  test:{len(test_df)}")
    print(f"  Seqs  → train:{X_train.shape[0]}  val:{X_val.shape[0]}  test:{X_test.shape[0]}")
    print(f"  Shape → X:{X_train.shape}  y:{y_train.shape}")
    print(f"  Scalers saved → models/{symbol}/")

print(f"\n── prepared built for {len(prepared)} symbols ──")


── 4080 ────────────────────────────────────────────────────────────
  Rows  → train:3696  val:616  test:617
  Seqs  → train:3666  val:586  test:587
  Shape → X:(3666, 30, 42)  y:(3666, 4)
  Scalers saved → models/4080/

── 2010 ────────────────────────────────────────────────────────────
  Rows  → train:4068  val:678  test:679
  Seqs  → train:4038  val:648  test:649
  Shape → X:(4038, 30, 42)  y:(4038, 4)
  Scalers saved → models/2010/

── 4170 ────────────────────────────────────────────────────────────
  Rows  → train:3768  val:628  test:628
  Seqs  → train:3738  val:598  test:598
  Shape → X:(3738, 30, 42)  y:(3738, 4)
  Scalers saved → models/4170/

── 4240 ────────────────────────────────────────────────────────────
  Rows  → train:3225  val:537  test:538
  Seqs  → train:3195  val:507  test:508
  Shape → X:(3195, 30, 42)  y:(3195, 4)
  Scalers saved → models/4240/

── prepared built for 4 symbols ──


In [5]:
import torch
from torch.utils.data import Dataset, DataLoader
import numpy as np


# ── Dataset ───────────────────────────────────────────────────────────────────
class TadawulDataset(Dataset):
    def __init__(self, X, y, weights=None):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)
        self.w = torch.tensor(weights, dtype=torch.float32) \
                 if weights is not None \
                 else torch.ones(len(X), dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx], self.w[idx]


BATCH_SIZE = 32
loaders    = {}  # loaders["4030"] → {train, val, test}

for symbol, data in prepared.items():
    train_loader = DataLoader(TadawulDataset(data["X_train"], data["y_train"]), batch_size=BATCH_SIZE, shuffle=True)
    val_loader   = DataLoader(TadawulDataset(data["X_val"],   data["y_val"]),   batch_size=BATCH_SIZE, shuffle=False)
    test_loader  = DataLoader(TadawulDataset(data["X_test"],  data["y_test"]),  batch_size=BATCH_SIZE, shuffle=False)

    loaders[symbol] = {
        "train": train_loader,
        "val":   val_loader,
        "test":  test_loader,
    }

    print(f"\n{symbol} — Train: {len(train_loader)} batches | Val: {len(val_loader)} batches | Test: {len(test_loader)} batches")

    # ── Sanity check ──────────────────────────────────────────────────────────
    sample_X, sample_y, sample_w = next(iter(train_loader))   # ← unpack 3
    print(f"  Sample X: {sample_X.shape} → (batch, seq_len, features)")
    print(f"  Sample y: {sample_y.shape} → (batch, 4)")
    print(f"  Sample w: {sample_w.shape} → (batch,)  weights all 1.0 since no weights passed")


4080 — Train: 115 batches | Val: 19 batches | Test: 19 batches
  Sample X: torch.Size([32, 30, 42]) → (batch, seq_len, features)
  Sample y: torch.Size([32, 4]) → (batch, 4)
  Sample w: torch.Size([32]) → (batch,)  weights all 1.0 since no weights passed

2010 — Train: 127 batches | Val: 21 batches | Test: 21 batches
  Sample X: torch.Size([32, 30, 42]) → (batch, seq_len, features)
  Sample y: torch.Size([32, 4]) → (batch, 4)
  Sample w: torch.Size([32]) → (batch,)  weights all 1.0 since no weights passed

4170 — Train: 117 batches | Val: 19 batches | Test: 19 batches
  Sample X: torch.Size([32, 30, 42]) → (batch, seq_len, features)
  Sample y: torch.Size([32, 4]) → (batch, 4)
  Sample w: torch.Size([32]) → (batch,)  weights all 1.0 since no weights passed

4240 — Train: 100 batches | Val: 16 batches | Test: 16 batches
  Sample X: torch.Size([32, 30, 42]) → (batch, seq_len, features)
  Sample y: torch.Size([32, 4]) → (batch, 4)
  Sample w: torch.Size([32]) → (batch,)  weights all 1.0 

In [6]:
import optuna
import torch
import torch.nn as nn
import numpy as np
from torch.utils.data import DataLoader, WeightedRandomSampler
from sklearn.metrics import balanced_accuracy_score
import torch.nn.functional as F

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

best_params_all = {}

# ── BiLSTM + Attention Model (with LayerNorm) ─────────────────────────────────
class StockPctChangeBiLSTMAttention(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, dropout=0.2):
        super().__init__()
        self.lstm      = nn.LSTM(input_size, hidden_size, num_layers,
                                 batch_first=True, bidirectional=True,
                                 dropout=dropout if num_layers > 1 else 0)
        # Attention with positional bias — later timesteps are more recent
        self.attention      = nn.Linear(hidden_size * 2, 1)
        self.pos_bias       = nn.Parameter(torch.zeros(1))  # learnable recency bias
        self.ln             = nn.LayerNorm(hidden_size * 2)
        self.dropout        = nn.Dropout(dropout)
        self.fc             = nn.Linear(hidden_size * 2, 4)
        
        # NEW: direct skip connection from last hidden to output
        self.skip_fc        = nn.Linear(hidden_size * 2, 4)
        self.output_blend   = nn.Parameter(torch.tensor(0.5))

    def forward(self, x):
        lstm_out, _  = self.lstm(x)                  # (B, T, H*2)
        seq_len      = lstm_out.size(1)
        
        # Recency-biased attention
        pos_weights  = torch.linspace(0, 1, seq_len, device=x.device).unsqueeze(-1)
        attn_raw     = self.attention(lstm_out) + self.pos_bias * pos_weights
        attn_weights = torch.softmax(attn_raw, dim=1)
        context      = (attn_weights * lstm_out).sum(dim=1)
        
        out_attn     = self.fc(self.dropout(self.ln(context)))
        
        # Skip: use last timestep directly (most recent information)
        last_hidden  = lstm_out[:, -1, :]
        out_skip     = self.skip_fc(last_hidden)
        
        # Blend — learnable, starts 50/50
        alpha        = torch.sigmoid(self.output_blend)
        return alpha * out_attn + (1 - alpha) * out_skip


# ── Joint Loss: Huber (magnitude) + Directional + Std-collapse penalty ────────
class JointPredictionLoss(nn.Module):
    def __init__(self, alpha=2.0, beta=0.5, gamma=1.0):
        super().__init__()
        self.alpha = alpha
        self.beta  = beta
        self.gamma = gamma  # NEW: magnitude-confidence penalty

    def forward(self, preds, targets, weights=None):
        # Huber for magnitude
        huber = nn.HuberLoss(delta=1.0, reduction="none")(preds, targets).mean(dim=1)

        # Directional loss — wrong sign with MAGNITUDE scaling
        # A prediction of -0.001 on +2.0 is penalized MORE than -0.001 on +0.1
        up_weight = torch.where(targets[:, 0] > 0,torch.full_like(targets[:, 0], 1.5),torch.full_like(targets[:, 0], 1.0))
        price_sign_loss = torch.clamp(-preds[:, 0] * targets[:, 0], min=0) * targets[:, 0].abs() 
        # price_sign_loss = torch.clamp(-preds[:, 0] * targets[:, 0], min=0) * targets[:, 0].abs() * up_weight
        volume_sign_loss = torch.clamp(-preds[:, 1] * targets[:, 1], min=0) * targets[:, 1].abs()
        high_sign_loss = torch.clamp(-preds[:, 2] * targets[:, 2], min=0) * targets[:,2].abs()
        low_sign_loss  = torch.clamp(-preds[:, 3] * targets[:, 3], min=0) * targets[:,3].abs()
        # Variance collapse penalty — directly penalize low prediction variance
        pred_std  = preds[:, 0].std() + 1e-8
        tgt_std   = targets[:, 0].std() + 1e-8
        std_ratio = pred_std / tgt_std
        std_match_loss = (pred_std - tgt_std).pow(2)
        # Penalize heavily when std_ratio < 0.3, ease off once > 0.7
        collapse_loss = torch.clamp(1.0 - std_ratio, min=0) ** 2

        # Bias penalty
        pred_up_frac = torch.sigmoid(preds[:, 0] * 3).mean()
        bias_penalty = (pred_up_frac - 0.5).pow(2) * 60.0
        # bias_penalty += (pred_up_frac - 0.5).abs() * 10.0
        price_dir_target = (targets[:, 0] > 0).float()
        
        bce_loss = F.binary_cross_entropy_with_logits(preds[:, 0], price_dir_target)
        

        per_sample = (
            huber
            + self.alpha * price_sign_loss
            + 0.3        * volume_sign_loss
            + 0.1        * high_sign_loss
            + 0.1        * low_sign_loss
            + 3.0        * std_match_loss
        )
        if weights is not None:
            per_sample = per_sample * weights

        return (
            per_sample.mean()
            + self.beta * (1.0 / std_ratio.clamp(min=0.05)).clamp(max=10.0)
            + self.gamma * collapse_loss * 5.0
            # + 0.0        * bias_penalty
            + 5.0        * bce_loss
            
            
        )
        
        
# ── Helpers ───────────────────────────────────────────────────────────────────
def create_sequences(X, y, seq_len, move_threshold=0.3):
    Xs, ys, ws = [], [], []
    for i in range(len(X) - seq_len):
        Xs.append(X[i : i + seq_len])
        ys.append(y[i + seq_len])
        move   = abs(y[i + seq_len, 0])
        ws.append(1.0 + 2.0 * float(move > move_threshold))
    return np.array(Xs), np.array(ys), np.array(ws, dtype=np.float32)


def make_balanced_sampler(y, oversample_factor=2.0):
    labels = (y[:, 0] > 0).astype(int)
    moves  = np.abs(y[:, 0])
    
    # Stratify into 4 buckets: UP-small, UP-large, DOWN-small, DOWN-large
    median_move = np.median(moves)
    bucket = labels * 2 + (moves > median_move).astype(int)
    # bucket: 0=DOWN-small, 1=DOWN-large, 2=UP-small, 3=UP-large
    
    counts = np.bincount(bucket, minlength=4).astype(float)
    counts = np.maximum(counts, 1)
    
    # Inverse frequency weights, with extra boost on large moves
    weight_map = 1.0 / counts
    weight_map[1] *= oversample_factor   # DOWN-large
    weight_map[3] *= oversample_factor   # UP-large
    
    sample_weights = torch.tensor(weight_map[bucket], dtype=torch.float32)
    return WeightedRandomSampler(sample_weights, len(sample_weights), replacement=True)


def safe_corr(a, b):
    """Pearson r, returns 0.0 on NaN / zero-std edge cases."""
    if np.std(a) < 1e-8 or np.std(b) < 1e-8:
        return 0.0
    r = np.corrcoef(a, b)[0, 1]
    return 0.0 if np.isnan(r) else float(r)


# ── Clean NaN values before Optuna ───────────────────────────────────────────
print("\nCleaning NaN values from all symbols...")
for symbol, data in prepared.items():
    for split in ("train", "val"):
        X_key, y_key = f"{split}_X_scaled", f"{split}_y_scaled"
        X, y         = data[X_key], data[y_key]
        mask         = ~(np.isnan(X).any(axis=1) | np.isnan(y).any(axis=1))
        before       = len(X)
        data[X_key]  = X[mask]
        data[y_key]  = y[mask]
        print(f"  {symbol} {split}: {before} → {len(data[X_key])} rows")


# ── Optuna search ─────────────────────────────────────────────────────────────
for symbol, data in prepared.items():
    print(f"\n{'═'*55}")
    print(f"  Optuna search — {symbol}")
    print(f"{'═'*55}")

    train_X    = data["train_X_scaled"]
    train_y    = data["train_y_scaled"]
    val_X      = data["val_X_scaled"]
    val_y      = data["val_y_scaled"]
    input_size = train_X.shape[1]

    def objective(trial):
        # ── Hyperparameters ──
        hidden_size    = trial.suggest_categorical("hidden_size", [64, 128])
        num_layers     = trial.suggest_int("num_layers", 2, 3)
        dropout        = trial.suggest_float("dropout", 0.2, 0.4, step=0.1)
        lr             = trial.suggest_float("lr", 1e-4, 5e-3, log=True)
        batch_size     = trial.suggest_categorical("batch_size", [32, 64])
        seq_len        = trial.suggest_categorical("seq_len", [10, 20, 30, 40])
        alpha          = trial.suggest_float("alpha", 3.0, 12.0, step=0.5)
        beta           = trial.suggest_float("beta", 0.3, 1.0, step=0.1)
        move_threshold = trial.suggest_float("move_threshold", 0.2, 0.5, step=0.1)
        gamma          = trial.suggest_float("gamma", 0.5, 3.0, step=0.5)

        # ── Sequences + loaders (built once, reused across seeds) ──
        X_tr, y_tr, w_tr = create_sequences(train_X, train_y, seq_len, move_threshold)
        X_vl, y_vl, _    = create_sequences(val_X,   val_y,   seq_len, move_threshold)

        sampler  = make_balanced_sampler(y_tr)
        train_ld = DataLoader(
            TadawulDataset(X_tr, y_tr, w_tr),
            batch_size=batch_size,
            sampler=sampler,
            drop_last=True
        )
        val_ld = DataLoader(
            TadawulDataset(X_vl, y_vl),
            batch_size=batch_size,
            shuffle=False,
            drop_last=False
        )

        # ── Multi-seed loop ──
        SEEDS       = [42, 123, 456]
        seed_scores = []

        for seed_idx, seed in enumerate(SEEDS):
            torch.manual_seed(seed)
            np.random.seed(seed)

            model = StockPctChangeBiLSTMAttention(
                input_size  = input_size,
                hidden_size = hidden_size,
                num_layers  = num_layers,
                dropout     = dropout
            ).to(device)

            criterion = JointPredictionLoss(alpha=alpha, beta=beta, gamma=gamma)
            optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)

            EPOCHS         = 50
            PATIENCE       = 10
            best_val_score = 0.0
            patience_ctr   = 0
            seed_collapsed = False

            for epoch in range(1, EPOCHS + 1):
                # ── Train ──
                model.train()
                for X_batch, y_batch, w_batch in train_ld:
                    X_batch = X_batch.to(device)
                    y_batch = y_batch.to(device)
                    w_batch = w_batch.to(device)
                    optimizer.zero_grad()
                    loss = criterion(model(X_batch), y_batch, w_batch)
                    loss.backward()
                    nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                    optimizer.step()

                # ── Validate ──
                model.eval()
                all_preds, all_actuals = [], []
                with torch.no_grad():
                    for X_batch, y_batch, _ in val_ld:
                        all_preds.append(model(X_batch.to(device)).cpu().numpy())
                        all_actuals.append(y_batch.numpy())

                if len(all_preds) == 0:
                    seed_collapsed = True
                    break

                val_preds   = np.vstack(all_preds)
                val_actuals = np.vstack(all_actuals)

                # ── Direction (balanced accuracy) ──
                price_dir = balanced_accuracy_score(
                    np.sign(val_actuals[:, 0]).astype(int),
                    np.sign(val_preds[:, 0]).astype(int)
                )
                volume_dir = balanced_accuracy_score(
                    np.sign(val_actuals[:, 1]).astype(int),
                    np.sign(val_preds[:, 1]).astype(int)
                )

                # ── Magnitude correlation ──
                price_corr  = safe_corr(val_actuals[:, 0], val_preds[:, 0])
                volume_corr = safe_corr(val_actuals[:, 1], val_preds[:, 1])

                # ── Std-ratio + collapse ──
                pred_std_ratio = np.std(val_preds[:, 0]) / max(np.std(val_actuals[:, 0]),
    1e-6)
                pred_up_pct    = (val_preds[:, 0] > 0).mean()
                actual_up_pct  = (val_actuals[:, 0] > 0).mean()
                bias_gap       = abs(pred_up_pct - actual_up_pct)

                if epoch >= 5:
                    if pred_up_pct < 0.15 or pred_up_pct > 0.85:
                        seed_collapsed = True
                        break
                    if pred_std_ratio < 0.20:
                        seed_collapsed = True
                        break

                balance_penalty = (pred_up_pct - 0.5) ** 2 * 8.0

                val_score = (
                    0.40 * price_dir +
                    0.20 * volume_dir +
                    0.25 * max(price_corr, 0.0) +
                    0.15 * max(volume_corr, 0.0) -
                    abs(1.0 - pred_std_ratio) * 0.3 -
                    bias_gap * 3.0 -
                    balance_penalty
                )

                if val_score > best_val_score:
                    best_val_score = val_score
                    patience_ctr   = 0
                else:
                    patience_ctr += 1
                    if patience_ctr >= PATIENCE:
                        break

            # ── End of epoch loop — record this seed's result ──
            seed_scores.append(-1.0 if seed_collapsed else best_val_score)

            # ── Optuna pruning per seed ──
            trial.report(float(np.mean(seed_scores)), seed_idx)
            if trial.should_prune():
                raise optuna.exceptions.TrialPruned()

        # ── Return mean across seeds ──
        return float(np.mean(seed_scores))

    # ── Run ───────────────────────────────────────────────────────────────────
    study = optuna.create_study(
        direction="maximize",
        sampler=optuna.samplers.TPESampler(seed=42),
        pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=5),
        study_name=f"bilstm_attention_{symbol}"
    )
    study.optimize(objective, n_trials=50, timeout=3600)

    best_params_all[symbol] = study.best_params

    # ── Results ───────────────────────────────────────────────────────────────
    print(f"\n  Best joint score : {study.best_value:.4f}")
    print(f"  Best trial       : {study.best_trial.number}")
    print(f"  Best params:")
    for k, v in study.best_params.items():
        print(f"    {k:15s} : {v}")

    print(f"\n  Top 5 Trials:")
    trials_df = study.trials_dataframe().sort_values("value", ascending=False).head()
    cols = [
        "number", "value",
        "params_hidden_size", "params_num_layers", "params_dropout",
        "params_lr", "params_batch_size", "params_seq_len",
        "params_alpha", "params_beta", "params_move_threshold",
    ]
    print(trials_df[[c for c in cols if c in trials_df.columns]])

    print(f"\n── Optuna done for {len(best_params_all)} symbols ──")
    print(trials_df[[c for c in cols if c in trials_df.columns]])

    print(f"\n── Optuna done for {len(best_params_all)} symbols ──")

[I 2026-06-03 17:10:27,965] A new study created in memory with name: bilstm_attention_4080


Using device: cuda

Cleaning NaN values from all symbols...
  4080 train: 3696 → 3696 rows
  4080 val: 616 → 616 rows
  2010 train: 4068 → 4068 rows
  2010 val: 678 → 678 rows
  4170 train: 3768 → 3768 rows
  4170 val: 628 → 628 rows
  4240 train: 3225 → 3225 rows
  4240 val: 537 → 537 rows

═══════════════════════════════════════════════════════
  Optuna search — 4080
═══════════════════════════════════════════════════════


[I 2026-06-03 17:11:16,442] Trial 0 finished with value: -0.2675274393694005 and parameters: {'hidden_size': 128, 'num_layers': 3, 'dropout': 0.30000000000000004, 'lr': 0.00018410729205738696, 'batch_size': 32, 'seq_len': 10, 'alpha': 12.0, 'beta': 0.9000000000000001, 'move_threshold': 0.2, 'gamma': 1.0}. Best is trial 0 with value: -0.2675274393694005.
[I 2026-06-03 17:11:49,676] Trial 1 finished with value: -1.0 and parameters: {'hidden_size': 128, 'num_layers': 3, 'dropout': 0.30000000000000004, 'lr': 0.0003124565071260876, 'batch_size': 32, 'seq_len': 40, 'alpha': 4.5, 'beta': 0.7, 'move_threshold': 0.4, 'gamma': 0.5}. Best is trial 0 with value: -0.2675274393694005.
[I 2026-06-03 17:12:31,305] Trial 2 finished with value: -1.0 and parameters: {'hidden_size': 64, 'num_layers': 2, 'dropout': 0.4, 'lr': 0.0043709904681305065, 'batch_size': 32, 'seq_len': 20, 'alpha': 7.5, 'beta': 0.3, 'move_threshold': 0.5, 'gamma': 1.0}. Best is trial 0 with value: -0.2675274393694005.
[I 2026-06-03


  Best joint score : 0.2354
  Best trial       : 44
  Best params:
    hidden_size     : 128
    num_layers      : 2
    dropout         : 0.4
    lr              : 0.00012447801816483226
    batch_size      : 32
    seq_len         : 10
    alpha           : 12.0
    beta            : 0.3
    move_threshold  : 0.4
    gamma           : 1.5

  Top 5 Trials:
    number     value  params_hidden_size  params_num_layers  params_dropout  \
44      44  0.235356                 128                  2             0.4   
42      42  0.212257                 128                  2             0.4   
31      31  0.211441                 128                  2             0.4   
26      26  0.205281                 128                  2             0.4   
41      41  0.182330                 128                  2             0.4   

    params_lr  params_batch_size  params_seq_len  params_alpha  params_beta  \
44   0.000124                 32              10          12.0          0.3   
42   0

[I 2026-06-03 17:53:57,856] Trial 0 finished with value: -1.0 and parameters: {'hidden_size': 128, 'num_layers': 3, 'dropout': 0.30000000000000004, 'lr': 0.00018410729205738696, 'batch_size': 32, 'seq_len': 10, 'alpha': 12.0, 'beta': 0.9000000000000001, 'move_threshold': 0.2, 'gamma': 1.0}. Best is trial 0 with value: -1.0.
[I 2026-06-03 17:54:34,157] Trial 1 finished with value: -1.0 and parameters: {'hidden_size': 128, 'num_layers': 3, 'dropout': 0.30000000000000004, 'lr': 0.0003124565071260876, 'batch_size': 32, 'seq_len': 40, 'alpha': 4.5, 'beta': 0.7, 'move_threshold': 0.4, 'gamma': 0.5}. Best is trial 0 with value: -1.0.
[I 2026-06-03 17:55:11,862] Trial 2 finished with value: -1.0 and parameters: {'hidden_size': 64, 'num_layers': 2, 'dropout': 0.4, 'lr': 0.0043709904681305065, 'batch_size': 32, 'seq_len': 20, 'alpha': 7.5, 'beta': 0.3, 'move_threshold': 0.5, 'gamma': 1.0}. Best is trial 0 with value: -1.0.
[I 2026-06-03 17:56:10,260] Trial 3 finished with value: -0.5966147395353


  Best joint score : 0.1257
  Best trial       : 44
  Best params:
    hidden_size     : 64
    num_layers      : 2
    dropout         : 0.30000000000000004
    lr              : 0.0001727469527957088
    batch_size      : 32
    seq_len         : 10
    alpha           : 5.0
    beta            : 0.4
    move_threshold  : 0.4
    gamma           : 1.0

  Top 5 Trials:
    number     value  params_hidden_size  params_num_layers  params_dropout  \
44      44  0.125744                  64                  2             0.3   
20      20  0.042364                  64                  2             0.3   
39      39 -0.172686                  64                  2             0.3   
37      37 -0.182391                  64                  2             0.3   
42      42 -0.198470                  64                  2             0.3   

    params_lr  params_batch_size  params_seq_len  params_alpha  params_beta  \
44   0.000173                 32              10           5.0          

[I 2026-06-03 18:25:14,757] Trial 0 finished with value: -0.32988367026906495 and parameters: {'hidden_size': 128, 'num_layers': 3, 'dropout': 0.30000000000000004, 'lr': 0.00018410729205738696, 'batch_size': 32, 'seq_len': 10, 'alpha': 12.0, 'beta': 0.9000000000000001, 'move_threshold': 0.2, 'gamma': 1.0}. Best is trial 0 with value: -0.32988367026906495.
[I 2026-06-03 18:25:57,693] Trial 1 finished with value: -0.6666666666666666 and parameters: {'hidden_size': 128, 'num_layers': 3, 'dropout': 0.30000000000000004, 'lr': 0.0003124565071260876, 'batch_size': 32, 'seq_len': 40, 'alpha': 4.5, 'beta': 0.7, 'move_threshold': 0.4, 'gamma': 0.5}. Best is trial 0 with value: -0.32988367026906495.
[I 2026-06-03 18:26:25,879] Trial 2 finished with value: -1.0 and parameters: {'hidden_size': 64, 'num_layers': 2, 'dropout': 0.4, 'lr': 0.0043709904681305065, 'batch_size': 32, 'seq_len': 20, 'alpha': 7.5, 'beta': 0.3, 'move_threshold': 0.5, 'gamma': 1.0}. Best is trial 0 with value: -0.3298836702690


  Best joint score : 0.0296
  Best trial       : 16
  Best params:
    hidden_size     : 128
    num_layers      : 3
    dropout         : 0.4
    lr              : 0.0001038206999168417
    batch_size      : 32
    seq_len         : 20
    alpha           : 10.5
    beta            : 0.8
    move_threshold  : 0.2
    gamma           : 1.5

  Top 5 Trials:
    number     value  params_hidden_size  params_num_layers  params_dropout  \
16      16  0.029644                 128                  3             0.4   
29      29 -0.198006                 128                  2             0.4   
22      22 -0.198257                 128                  2             0.4   
41      41 -0.204779                 128                  2             0.4   
20      20 -0.222146                 128                  2             0.4   

    params_lr  params_batch_size  params_seq_len  params_alpha  params_beta  \
16   0.000104                 32              20          10.5          0.8   
29   0.

[I 2026-06-03 19:00:51,794] Trial 0 finished with value: -1.0 and parameters: {'hidden_size': 128, 'num_layers': 3, 'dropout': 0.30000000000000004, 'lr': 0.00018410729205738696, 'batch_size': 32, 'seq_len': 10, 'alpha': 12.0, 'beta': 0.9000000000000001, 'move_threshold': 0.2, 'gamma': 1.0}. Best is trial 0 with value: -1.0.
[I 2026-06-03 19:01:15,889] Trial 1 finished with value: -1.0 and parameters: {'hidden_size': 128, 'num_layers': 3, 'dropout': 0.30000000000000004, 'lr': 0.0003124565071260876, 'batch_size': 32, 'seq_len': 40, 'alpha': 4.5, 'beta': 0.7, 'move_threshold': 0.4, 'gamma': 0.5}. Best is trial 0 with value: -1.0.
[I 2026-06-03 19:01:39,552] Trial 2 finished with value: -1.0 and parameters: {'hidden_size': 64, 'num_layers': 2, 'dropout': 0.4, 'lr': 0.0043709904681305065, 'batch_size': 32, 'seq_len': 20, 'alpha': 7.5, 'beta': 0.3, 'move_threshold': 0.5, 'gamma': 1.0}. Best is trial 0 with value: -1.0.
[I 2026-06-03 19:02:03,820] Trial 3 finished with value: -1.0 and paramet


  Best joint score : -1.0000
  Best trial       : 0
  Best params:
    hidden_size     : 128
    num_layers      : 3
    dropout         : 0.30000000000000004
    lr              : 0.00018410729205738696
    batch_size      : 32
    seq_len         : 10
    alpha           : 12.0
    beta            : 0.9000000000000001
    move_threshold  : 0.2
    gamma           : 1.0

  Top 5 Trials:
    number  value  params_hidden_size  params_num_layers  params_dropout  \
0        0   -1.0                 128                  3             0.3   
37      37   -1.0                 128                  3             0.3   
27      27   -1.0                 128                  3             0.4   
28      28   -1.0                  64                  2             0.4   
29      29   -1.0                  64                  3             0.2   

    params_lr  params_batch_size  params_seq_len  params_alpha  params_beta  \
0    0.000184                 32              10          12.0          

In [7]:
# Training 
import torch
import torch.nn as nn
from torch.optim.lr_scheduler import ReduceLROnPlateau
from sklearn.metrics import balanced_accuracy_score
import numpy as np
import os
import time

trained_models = {}

def compute_joint_score(price_dir, volume_dir, price_corr, volume_corr,
                          pred_up_pct, std_ratio):
      bias_gap        = abs(pred_up_pct - 0.5)
      std_penalty     = abs(1.0 - std_ratio) * 0.3
      balance_penalty = (pred_up_pct - 0.5) ** 2 * 8.0
      return (
          0.40 * price_dir +
          0.20 * volume_dir +
          0.25 * max(price_corr,  0.0) +
          0.15 * max(volume_corr, 0.0) -
          std_penalty - bias_gap * 3.0 - balance_penalty
      )


# ── Validation metrics: direction + magnitude ─────────────────────────────────
def compute_val_metrics(model, loader, device):
    """
    Returns
    -------
    joint_score  : float  — same formula as Optuna objective
    price_dir    : float  — balanced accuracy, price direction
    volume_dir   : float  — balanced accuracy, volume direction
    price_corr   : float  — Pearson r, price magnitude
    volume_corr  : float  — Pearson r, volume magnitude
    pred_up_pct  : float  — fraction of UP predictions (collapse monitor)
    std_ratio    : float  — pred_std / actual_std  (1.0 = perfect)
    """
    model.eval()
    all_preds, all_actuals = [], []
    with torch.no_grad():
        for X_batch, y_batch, _ in loader:    # ← unpack 3
            all_preds.append(model(X_batch.to(device)).cpu().numpy())
            all_actuals.append(y_batch.numpy())

    preds   = np.vstack(all_preds)
    actuals = np.vstack(all_actuals)

    # ── Directional (balanced accuracy) ──────────────────────────────────────
    price_dir  = balanced_accuracy_score(
        np.sign(actuals[:, 0]).astype(int),
        np.sign(preds[:, 0]).astype(int)
    )
    volume_dir = balanced_accuracy_score(
        np.sign(actuals[:, 1]).astype(int),
        np.sign(preds[:, 1]).astype(int)
    )

    # ── Magnitude correlation (Pearson r) ─────────────────────────────────────
    def safe_corr(a, b):
        if np.std(a) < 1e-8 or np.std(b) < 1e-8:
            return 0.0
        r = np.corrcoef(a, b)[0, 1]
        return 0.0 if np.isnan(r) else float(r)

    price_corr  = safe_corr(actuals[:, 0], preds[:, 0])
    volume_corr = safe_corr(actuals[:, 1], preds[:, 1])

    # ── Std-ratio and collapse ────────────────────────────────────────────────
    pred_up_pct  = (preds[:, 0] > 0).mean()
    std_ratio    = np.std(preds[:, 0]) / max(np.std(actuals[:, 0]), 1e-6)
    std_penalty  = abs(1.0 - std_ratio) * 0.2
    collapse_penalty = 1.0 if (pred_up_pct < 0.05 or pred_up_pct > 0.95) else 0.0
    
    

    # ── Joint score (mirrors Optuna objective exactly) ────────────────────────
    joint_score = (
        0.35 * price_dir               +
        0.25 * volume_dir              +
        0.25 * max(price_corr,  0.0)   +
        0.15 * max(volume_corr, 0.0)   -
        std_penalty                    -
        collapse_penalty
    )
    

    return joint_score, price_dir, volume_dir, price_corr, volume_corr, pred_up_pct, std_ratio


# ── Training loop ─────────────────────────────────────────────────────────────
for symbol, data in prepared.items():
    print(f"\n{'═'*72}")
    print(f"  Training — {symbol}")
    print(f"{'═'*72}")

    best = best_params_all[symbol]
    print(f"  Best params: {best}")

    move_threshold = best.get("move_threshold", 0.3)   # fallback if not in best_params

    # ── Rebuild sequences with best seq_len ───────────────────────────────────
    X_train_f, y_train_f, w_train_f = create_sequences(data["train_X_scaled"], data["train_y_scaled"], best["seq_len"], move_threshold)
    X_val_f,   y_val_f,   _         = create_sequences(data["val_X_scaled"],   data["val_y_scaled"],   best["seq_len"], move_threshold)
    X_test_f,  y_test_f,  _         = create_sequences(data["test_X_scaled"],  data["test_y_scaled"],  best["seq_len"], move_threshold)

    # ── Loaders ───────────────────────────────────────────────────────────────
    sampler      = make_balanced_sampler(y_train_f)
    train_loader = DataLoader(
        TadawulDataset(X_train_f, y_train_f, w_train_f),   # ← pass weights
        batch_size=best["batch_size"],
        sampler=sampler,
        drop_last=True
    )
    val_loader  = DataLoader(TadawulDataset(X_val_f,  y_val_f),  batch_size=best["batch_size"], shuffle=False)
    test_loader = DataLoader(TadawulDataset(X_test_f, y_test_f), batch_size=best["batch_size"], shuffle=False)

    # ── Model ─────────────────────────────────────────────────────────────────
    model = StockPctChangeBiLSTMAttention(
        input_size  = data["train_X_scaled"].shape[1],
        hidden_size = best["hidden_size"],
        num_layers  = best["num_layers"],
        dropout     = best["dropout"]
    ).to(device)

    WARMUP_EPOCHS    = 10
    BASE_ALPHA       = best["alpha"]
    BASE_BETA        = best["beta"]
    BASE_GAMMA       = best.get("gamma", 1.0)   # fallback for old best_params
    optimizer        = torch.optim.Adam(model.parameters(), lr=best["lr"], weight_decay=1e-5)
    scheduler        = ReduceLROnPlateau(optimizer, mode='max', patience=5, factor=0.5)
    #                                          ↑ 'max' — we track joint_score, not loss

    EPOCHS           = 100
    PATIENCE         = 15
    best_joint_score = -np.inf
    patience_ctr     = 0
    history          = {"train_loss": [], "val_loss": [], "joint_score": [], "price_dir": [], "price_corr": []}

    os.makedirs(f"models/{symbol}", exist_ok=True)
    save_path = f"models/{symbol}/{symbol}_best_bilstm.pt"

    print(f"\n{'Epoch':>6} | {'TrLoss':>8} | {'VaLoss':>8} | "
          f"{'Joint':>7} | {'PDir':>6} | {'VDir':>6} | "
          f"{'Pr':>6} | {'Vr':>6} | {'UP%':>5} | {'StdR':>5} | {'s':>5}")
    print("─" * 90)
    
    training_state = type('State', (), {})() 
    
    for epoch in range(1, EPOCHS + 1):
        start        = time.time()
        warmup_scale = 0.3 + 0.7 * min(1.0, epoch / WARMUP_EPOCHS) 
        criterion    = JointPredictionLoss(
            alpha = BASE_ALPHA * warmup_scale,
            beta  = BASE_BETA,
            gamma = BASE_GAMMA
        )

        # ── Train ─────────────────────────────────────────────────────────────
        model.train()
        train_losses = []
        for X_batch, y_batch, w_batch in train_loader:
            X_batch, y_batch, w_batch = X_batch.to(device), y_batch.to(device), w_batch.to(device)
            optimizer.zero_grad()
            loss = criterion(model(X_batch), y_batch, w_batch)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            train_losses.append(loss.item())

        # ── Val loss ──────────────────────────────────────────────────────────
        model.eval()
        val_losses = []
        with torch.no_grad():
            for X_batch, y_batch, _ in val_loader:
                val_losses.append(
                    criterion(model(X_batch.to(device)), y_batch.to(device)).item()
                )

        train_loss = np.mean(train_losses)
        val_loss   = np.mean(val_losses)

        joint_score, p_dir, v_dir, p_corr, v_corr, pred_up_pct, std_ratio = \
            compute_val_metrics(model, val_loader, device)

        elapsed = time.time() - start
        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["joint_score"].append(joint_score)
        history["price_dir"].append(p_dir)
        history["price_corr"].append(p_corr)

        # ── Print once ────────────────────────────────────────────────────────
        print(f"{epoch:>6} | {train_loss:>8.5f} | {val_loss:>8.5f} | "
              f"{joint_score:>7.4f} | {p_dir:>6.3f} | {v_dir:>6.3f} | "
              f"{p_corr:>6.3f} | {v_corr:>6.3f} | {pred_up_pct:>4.0%} | "
              f"{std_ratio:>5.2f} | {elapsed:>4.1f}s")

        # ── Collapse detection with restart ───────────────────────────────────
        if epoch > WARMUP_EPOCHS:
            if pred_up_pct > 0.85 or pred_up_pct < 0.15 or std_ratio < 0.20:
                consecutive_collapse = getattr(training_state, 'collapse_ctr', 0) + 1
                training_state.collapse_ctr = consecutive_collapse
                if consecutive_collapse >= 5:
                    print(f"\n  Collapse detected at epoch {epoch} "
                            f"(up%={pred_up_pct:.0%}, std_r={std_ratio:.2f}) — resetting weights")
                    model.apply(lambda m: m.reset_parameters()
                                if hasattr(m, 'reset_parameters') else None)
                    for pg in optimizer.param_groups:
                        pg['lr'] = best["lr"]
                    training_state.collapse_ctr = 0
            else:
                training_state.collapse_ctr = 0

        # ── Scheduler — once, after collapse check ────────────────────────────
        scheduler.step(joint_score)

        # ── Save on best joint score ───────────────────────────────────────────
        if joint_score > best_joint_score:
            best_joint_score = joint_score
            patience_ctr     = 0
            torch.save(model.state_dict(), save_path)
            print(f"         ✓ Saved  "
                  f"(p_dir={p_dir:.3f}, v_dir={v_dir:.3f}, "
                  f"p_corr={p_corr:.3f}, up%={pred_up_pct:.0%}, std_r={std_ratio:.2f})")
        else:
            patience_ctr += 1
            if patience_ctr >= PATIENCE:
                print(f"\n  Early stopping at epoch {epoch}.")
                break

    trained_models[symbol] = {
        "best_joint_score": best_joint_score,
        "history":          history,
    }
    print(f"\n  Best joint score: {best_joint_score:.4f}")

print(f"\n── Training done for {len(trained_models)} symbols ──")
for s, r in trained_models.items():
    print(f"  {s}: best joint score = {r['best_joint_score']:.4f}")


════════════════════════════════════════════════════════════════════════
  Training — 4080
════════════════════════════════════════════════════════════════════════
  Best params: {'hidden_size': 128, 'num_layers': 2, 'dropout': 0.4, 'lr': 0.00012447801816483226, 'batch_size': 32, 'seq_len': 10, 'alpha': 12.0, 'beta': 0.3, 'move_threshold': 0.4, 'gamma': 1.5}

 Epoch |   TrLoss |   VaLoss |   Joint |   PDir |   VDir |     Pr |     Vr |   UP% |  StdR |     s
──────────────────────────────────────────────────────────────────────────────────────────
     1 | 13.27693 |  7.82728 |  0.3122 |  0.504 |  0.504 |  0.057 |  0.025 |  30% |  1.04 |  2.6s
         ✓ Saved  (p_dir=0.504, v_dir=0.504, p_corr=0.057, up%=30%, std_r=1.04)
     2 | 12.44333 |  7.65631 |  0.3027 |  0.516 |  0.505 |  0.065 |  0.047 |  19% |  1.14 |  2.5s
     3 | 13.04102 |  7.48834 |  0.3137 |  0.512 |  0.510 |  0.062 |  0.049 |  28% |  1.08 |  2.6s
         ✓ Saved  (p_dir=0.512, v_dir=0.510, p_corr=0.062, up%=28%, std_r

DIAGNOSTIC

In [8]:
# ── Evaluation on Test Set ────────────────────────────────────────────────────
print("\n" + "=" * 50)
print("EVALUATION ON TEST SET")
print("=" * 50)

def evaluate(actuals, preds, label):
    mae      = np.mean(np.abs(actuals - preds))
    rmse     = np.sqrt(np.mean((actuals - preds) ** 2))
    dir_acc  = np.mean(np.sign(actuals) == np.sign(preds)) * 100

    corr       = np.corrcoef(actuals, preds)[0, 1]
    pred_std   = np.std(preds)
    actual_std = np.std(actuals)
    within_1pct = np.mean(np.abs(actuals - preds) < 1.0) * 100
    within_2pct = np.mean(np.abs(actuals - preds) < 2.0) * 100

    print(f"\n  [{label}]")
    print(f"  MAE             : {mae:.4f}%")
    print(f"  RMSE            : {rmse:.4f}%")
    print(f"  Pearson r       : {corr:.4f}")
    print(f"  Pred std        : {pred_std:.4f}  Actual std: {actual_std:.4f}")
    print(f"  Within ±1%      : {within_1pct:.1f}%")
    print(f"  Within ±2%      : {within_2pct:.1f}%")
    print(f"  Directional Acc : {dir_acc:.2f}%")

for symbol in symbols:
    if symbol not in prepared:
        print(f"\n  Skipping {symbol} — not in prepared dict.")
        continue
    if symbol not in trained_models:
        print(f"\n  Skipping {symbol} — no trained model found.")
        continue

    print(f"\n{'═'*55}")
    print(f"  Evaluating — {symbol}")
    print(f"{'═'*55}")

    best = best_params_all[symbol]
    data = prepared[symbol]
    move_threshold = best.get("move_threshold", 0.3)

    # ── Rebuild test loader with best seq_len ─────────────────────────────────
    X_test_f, y_test_f, _ = create_sequences(          # ← unpack 3, discard w
        data["test_X_scaled"], data["test_y_scaled"], best["seq_len"], move_threshold
    )
    test_loader = DataLoader(
        TadawulDataset(X_test_f, y_test_f),            # no weights needed for eval
        batch_size=best["batch_size"],
        shuffle=False
    )

    # ── Load best model ───────────────────────────────────────────────────────
    checkpoint = torch.load(
        f"models/{symbol}/{symbol}_best_bilstm.pt",
        weights_only=True
    )

    inferred_hidden_size = checkpoint["lstm.weight_ih_l0"].shape[0] // 4
    inferred_num_layers  = sum(
        1 for k in checkpoint if k.startswith("lstm.weight_ih_l") and "_reverse" not in k
    )

    model = StockPctChangeBiLSTMAttention(
        input_size  = data["train_X_scaled"].shape[1],
        hidden_size = inferred_hidden_size,
        num_layers  = inferred_num_layers,
        dropout     = best["dropout"]
    ).to(device)

    model.load_state_dict(checkpoint)
    model.eval()

    # ── Load target scaler ────────────────────────────────────────────────────
    with open(f"models/{symbol}/{symbol}_target_scaler.pkl", "rb") as f:
        target_scaler = pickle.load(f)

    # ── Inference ─────────────────────────────────────────────────────────────
    all_preds, all_actuals = [], []
    with torch.no_grad():
        for X_batch, y_batch, _ in test_loader:        # ← unpack 3
            preds   = model(X_batch.to(device)).cpu().numpy()
            actuals = y_batch.numpy()
            all_preds.append(preds)
            all_actuals.append(actuals)

    all_preds   = np.vstack(all_preds)
    all_actuals = np.vstack(all_actuals)

    # ── Inverse transform ─────────────────────────────────────────────────────
    all_preds   = target_scaler.inverse_transform(all_preds)
    all_actuals = target_scaler.inverse_transform(all_actuals)

    price_preds_pct    = all_preds[:, 0]
    price_actuals_pct  = all_actuals[:, 0]
    volume_preds_pct   = np.expm1(all_preds[:, 1])   * 100
    volume_actuals_pct = np.expm1(all_actuals[:, 1]) * 100
    
    high_preds_pct     = all_preds[:, 2] * 100
    high_actuals_pct   = all_actuals[:, 2] * 100
    low_preds_pct      = all_preds[:, 3] * 100
    low_actuals_pct    = all_actuals[:, 3] * 100

    # ── Metrics ───────────────────────────────────────────────────────────────
    evaluate(price_actuals_pct,  price_preds_pct,  "Price % Change")
    evaluate(volume_actuals_pct, volume_preds_pct, "Volume % Change")
    evaluate(high_actuals_pct,   high_preds_pct,   "High % Change")   # add
    evaluate(low_actuals_pct,    low_preds_pct,    "Low % Change") 

    # ── Save predictions ──────────────────────────────────────────────────────
    results_df = pd.DataFrame({
        "actual_price_pct"    : price_actuals_pct,
        "predicted_price_pct" : price_preds_pct,
        "actual_volume_pct"   : volume_actuals_pct,
        "predicted_volume_pct": volume_preds_pct,
        "actual_high_pct"     : high_actuals_pct,
        "predicted_high_pct"  : high_preds_pct,
        "actual_low_pct"      : low_actuals_pct,
        "predicted_low_pct"   : low_preds_pct,
    })
    out_path = f"models/{symbol}/{symbol}_test_predictions.csv"
    results_df.to_csv(out_path, index=False)
    print(f"\n  Predictions saved: {out_path}")

print(f"\n── Evaluation done for {len(symbols)} symbols ──")


EVALUATION ON TEST SET

═══════════════════════════════════════════════════════
  Evaluating — 4080
═══════════════════════════════════════════════════════

  [Price % Change]
  MAE             : 2.2635%
  RMSE            : 2.8043%
  Pearson r       : 0.0654
  Pred std        : 1.9178  Actual std: 1.5776
  Within ±1%      : 28.2%
  Within ±2%      : 49.8%
  Directional Acc : 53.21%

  [Volume % Change]
  MAE             : 49.1493%
  RMSE            : 66.8220%
  Pearson r       : 0.0842
  Pred std        : 5.8934  Actual std: 66.2993
  Within ±1%      : 0.2%
  Within ±2%      : 2.6%
  Directional Acc : 53.54%

  [High % Change]
  MAE             : 1.0641%
  RMSE            : 1.4218%
  Pearson r       : 0.0355
  Pred std        : 0.1804  Actual std: 1.4089
  Within ±1%      : 55.8%
  Within ±2%      : 86.0%
  Directional Acc : 55.35%

  [Low % Change]
  MAE             : 1.1129%
  RMSE            : 1.4804%
  Pearson r       : 0.0625
  Pred std        : 0.1579  Actual std: 1.4819
  Withi

In [9]:
import pandas as pd
import numpy as np

for symbol in symbols:
    csv_path = f"models/{symbol}/{symbol}_test_predictions.csv"

    try:
        results = pd.read_csv(csv_path)
    except FileNotFoundError:
        print(f"\n  Skipping {symbol} — {csv_path} not found.")
        continue

    price_actual    = results["actual_price_pct"]
    price_predicted = results["predicted_price_pct"]

    correct   = (np.sign(price_actual) == np.sign(price_predicted))
    up_mask   = price_actual > 0
    down_mask = price_actual < 0

    print(f"\n{'═'*55}")
    print(f"  {symbol} — Honest Model Baseline")
    print(f"{'═'*55}")
    print(f"  Actual    mean : {price_actual.mean():.4f}   std: {price_actual.std():.4f}")
    print(f"  Predicted mean : {price_predicted.mean():.4f}   std: {price_predicted.std():.4f}")
    print(f"\n  Overall Dir Acc : {correct.mean()*100:.2f}%")
    print(f"  When actual UP  : {correct[up_mask].mean()*100:.2f}%  ({up_mask.sum()} samples)")
    print(f"  When actual DOWN: {correct[down_mask].mean()*100:.2f}%  ({down_mask.sum()} samples)")
    print(f"\n  % predicted UP  : {(price_predicted > 0).mean()*100:.2f}%")
    print(f"  % actual UP     : {(price_actual > 0).mean()*100:.2f}%")

    print(f"\n  Dir Acc by Move Size:")
    bins   = [0, 0.5, 1.0, 2.0, 5.0, 100.0]
    labels = ["<0.5%", "0.5-1%", "1-2%", "2-5%", ">5%"]
    price_actual_abs = price_actual.abs()
    for i, label in enumerate(labels):
        mask = (price_actual_abs >= bins[i]) & (price_actual_abs < bins[i+1])
        if mask.sum() > 0:
            acc = correct[mask].mean() * 100
            print(f"    {label:>8} moves : {acc:.2f}%  ({mask.sum()} samples)")

print(f"\n── Diagnostics done for {len(symbols)} symbols ──")


═══════════════════════════════════════════════════════
  4080 — Honest Model Baseline
═══════════════════════════════════════════════════════
  Actual    mean : -0.0091   std: 1.5789
  Predicted mean : -1.4559   std: 1.9194

  Overall Dir Acc : 53.21%
  When actual UP  : 34.23%  (298 samples)
  When actual DOWN: 71.52%  (309 samples)

  % predicted UP  : 31.30%
  % actual UP     : 49.09%

  Dir Acc by Move Size:
       <0.5% moves : 52.43%  (185 samples)
      0.5-1% moves : 53.03%  (132 samples)
        1-2% moves : 55.95%  (168 samples)
        2-5% moves : 50.83%  (120 samples)
         >5% moves : 50.00%  (2 samples)

═══════════════════════════════════════════════════════
  2010 — Honest Model Baseline
═══════════════════════════════════════════════════════
  Actual    mean : -0.0863   std: 1.1183
  Predicted mean : 1.4327   std: 1.1622

  Overall Dir Acc : 51.87%
  When actual UP  : 85.16%  (337 samples)
  When actual DOWN: 18.07%  (332 samples)

  % predicted UP  : 83.56%
  % 

In [10]:
for symbol in symbols:
    df = pd.read_csv(f'{symbol}_daily.csv', parse_dates=["datetime"])
    n = len(df)
    val_df  = df.iloc[int(n*0.70):int(n*0.85)]
    test_df = df.iloc[int(n*0.85):]
    
    print(f"\n{symbol}")
    print(f"  Val  UP%: {(val_df['price_pct_change'] > 0).mean():.2%}  ({val_df['datetime'].min().date()} → {val_df['datetime'].max().date()})")
    print(f"  Test UP%: {(test_df['price_pct_change'] > 0).mean():.2%}  ({test_df['datetime'].min().date()} → {test_df['datetime'].max().date()})")


4080
  Val  UP%: 45.55%  (2019-09-16 → 2023-01-09)
  Test UP%: 45.19%  (2023-01-10 → 2026-05-13)

2010
  Val  UP%: 47.41%  (2019-01-28 → 2022-09-19)
  Test UP%: 44.84%  (2022-09-20 → 2026-05-13)

4170
  Val  UP%: 49.16%  (2019-09-11 → 2023-01-08)
  Test UP%: 33.81%  (2023-01-09 → 2026-05-13)

4240
  Val  UP%: 43.47%  (2020-08-06 → 2023-06-22)
  Test UP%: 41.94%  (2023-07-02 → 2026-05-13)
